# Experiment 1: Time-CAG v1 (Baseline Configuration)

**Objective:** Establish baseline Time-CAG performance using long context length

**Configuration:**
- Model: TransformerLongContext
- Dataset: ETTh1 (Electricity Transformer Temperature)
- Context Length (seq_len): 720 timesteps
- Prediction Length: 96 timesteps
- Model Dimension: 128
- Encoder Layers: 2
- Dropout: 0.1
- Patch Size: 12 (PatchTST-style)
- Batch Size: 16
- Training Epochs: 2 (quick test, increase for final run)

**Expected Outcome:** Baseline MSE/MAE for comparison with RAFT

---

## 📌 Instructions

**For Remote GPU:**
1. Upload this notebook to your GPU machine
2. Run all cells sequentially
3. Estimated time: ~30-60 minutes for 2 epochs

**For Local:**
- Ensure you're in `/Users/rishi/StudioProjects/caft/RAFT/`
- Virtual environment activated

## 1️⃣ Setup Environment

Clones RAFT repository and downloads datasets (runs only if RAFT doesn't exist yet).

In [ ]:
# Setup Environment
import os
import sys

# Check if RAFT already exists
if not os.path.exists('RAFT'):
    print("📥 Cloning RAFT repository...")
    
    # Install dependencies
    os.system('pip install -q torch torchvision sktime')
    
    # Clone RAFT repository
    os.system('git clone https://github.com/archon159/RAFT.git')
    os.chdir('RAFT')
    
    # Download ETT datasets
    os.makedirs('data/ETT', exist_ok=True)
    print("📥 Downloading ETT datasets...")
    os.system('wget -q -O data/ETT/ETTh1.csv https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv')
    os.system('wget -q -O data/ETT/ETTh2.csv https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh2.csv')
    os.system('wget -q -O data/ETT/ETTm1.csv https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv')
    os.system('wget -q -O data/ETT/ETTm2.csv https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm2.csv')
    
    print(f"✅ Setup complete! Working directory: {os.getcwd()}")
else:
    # RAFT already exists, just navigate to it
    os.chdir('RAFT')
    print(f"✅ RAFT directory found. Working directory: {os.getcwd()}")

# Check GPU availability
import torch
if torch.cuda.is_available():
    print(f"🎮 GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  No GPU detected - will use CPU (slow)")


## 2️⃣ Import Libraries and Setup Path

In [ ]:
import os
import sys
import json
import time
from datetime import datetime
import numpy as np
import torch

# Add RAFT root to Python path
raft_root = os.getcwd()
if raft_root not in sys.path:
    sys.path.insert(0, raft_root)

print(f"RAFT root: {raft_root}")
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

# Check device
if torch.cuda.is_available():
    device = 'cuda'
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = 'mps'
    print("GPU: Apple Silicon (MPS)")
else:
    device = 'cpu'
    print("GPU: None (using CPU)")

# Import RAFT modules
try:
    from data_provider.data_factory import data_provider
    from exp.exp_basic import Exp_Basic
    
    # Use Exp_Basic as the base (standard RAFT experiment class)
    Exp_Long_Context_Forecast = Exp_Basic
    
    print("✅ All imports successful!")
    print("✅ Using Exp_Basic from RAFT repository")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("\n⚠️  Make sure you're running from the RAFT directory")
    print("   and all required files are present.")
    raise

## 3️⃣ Configure Experiment Parameters

In [ ]:
class Args:
    """Experiment 1: Time-CAG v1 - Long Context"""
    
    # Basic Config
    task_name = 'long_term_forecast'
    is_training = 1
    model_id = 'ETTh1_exp1'
    model = 'TransformerLongContext'  # Our custom long context model
    
    # Data
    data = 'ETTh1'
    root_path = './data/ETT/'
    data_path = 'ETTh1.csv'
    features = 'M'
    target = 'OT'
    freq = 'h'
    checkpoints = './checkpoints/'
    
    # Forecasting Task
    seq_len = 720  # Start with 720 (same as RAFT), increase later
    label_len = 48
    pred_len = 96
    seasonal_patterns = 'Monthly'
    inverse = False
    
    # Model Architecture
    enc_in = 7
    dec_in = 7
    c_out = 7
    d_model = 128
    n_heads = 8
    e_layers = 2
    d_layers = 1
    d_ff = 512
    moving_avg = 25
    factor = 1
    distil = True
    dropout = 0.1
    embed = 'timeF'
    activation = 'gelu'
    output_attention = False
    
    # Patching
    patch_size = 12
    stride = 12
    
    # Training
    num_workers = 0
    itr = 1
    train_epochs = 2  # Quick test first
    batch_size = 16
    patience = 3
    learning_rate = 0.0001
    des = 'Exp1_LongContext'
    loss = 'MSE'
    lradj = 'type1'
    use_amp = False
    
    # GPU
    use_gpu = True
    gpu = 0
    use_multi_gpu = False
    devices = '0'

args = Args()

print("=" * 80)
print("EXPERIMENT 1: Long Context Transformer")
print("=" * 80)
print(f"Model: {args.model}")
print(f"Context: {args.seq_len} | Prediction: {args.pred_len}")
print(f"d_model: {args.d_model} | layers: {args.e_layers}")
print(f"Patch size: {args.patch_size} | Patches: {(args.seq_len // args.stride)}")
print(f"Batch: {args.batch_size} | Epochs: {args.train_epochs}")
print("=" * 80)

## 4️⃣ Initialize Experiment

Create the experiment instance which will handle data loading, model initialization, and training.

In [ ]:
# Initialize experiment
from exp.exp_basic import Exp_Basic

print("Initializing experiment...")
exp = Exp_Basic(args)

# Create experiment identifier
setting = f'{args.model_id}_{args.model}_{args.data}_{args.features}_sl{args.seq_len}_pl{args.pred_len}_dm{args.d_model}_el{args.e_layers}'

print(f"✅ Experiment initialized: {setting}")
print(f"Model device: {exp.device}")

## 5️⃣ Train the Model

This will take approximately **1-2 hours** depending on your GPU.

**Expected behavior:**
- 10 epochs of training
- Each epoch shows train loss and validation loss
- Model checkpoints saved to `./checkpoints/`

In [ ]:
print("=" * 80)
print("TRAINING PHASE")
print("=" * 80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# Start timing
start_time = time.time()

# Train the model
exp.train(setting)

# Calculate training time
training_time = time.time() - start_time

print()
print("=" * 80)
print(f"Training complete!")
print(f"Training time: {training_time/60:.2f} minutes ({training_time/3600:.2f} hours)")
print("=" * 80)

## 6️⃣ Evaluate on Test Set

Run the trained model on the test dataset to get final performance metrics.

In [ ]:
print("=" * 80)
print("TESTING PHASE")
print("=" * 80)

# Start timing
test_start = time.time()

# Run test
mse, mae = exp.test(setting, test=1)

# Calculate test time
test_time = time.time() - test_start
total_time = time.time() - start_time

print()
print("=" * 80)
print("TEST RESULTS")
print("=" * 80)
print(f"Test MSE: {mse:.6f}")
print(f"Test MAE: {mae:.6f}")
print(f"Test time: {test_time/60:.2f} minutes")
print(f"Total time: {total_time/60:.2f} minutes ({total_time/3600:.2f} hours)")
print("=" * 80)

# Store results for later use
results = {
    'experiment_id': 1,
    'experiment_name': 'Time-CAG v1',
    'status': 'SUCCESS',
    'test_mse': float(mse),
    'test_mae': float(mae),
    'training_time_minutes': round(training_time / 60, 2),
    'test_time_minutes': round(test_time / 60, 2),
    'total_time_minutes': round(total_time / 60, 2),
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

## 7️⃣ Save Results to JSON

Save experiment results for comparison with other experiments and graph generation.

In [ ]:
# Create results directory
results_dir = './experiments/results'
os.makedirs(results_dir, exist_ok=True)

# Save detailed results
results_file = os.path.join(results_dir, 'exp1_results.json')
with open(results_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✅ Results saved to: {results_file}")

# Also append to master log (for tracking all experiments)
master_log = os.path.join(results_dir, 'all_experiments.jsonl')
with open(master_log, 'a') as f:
    f.write(json.dumps(results) + '\n')

print(f"✅ Appended to master log: {master_log}")

# Display saved results
print("\n" + "=" * 80)
print("SAVED RESULTS")
print("=" * 80)
print(json.dumps(results, indent=2))
print("=" * 80)

## 8️⃣ Compare with RAFT Baseline

RAFT's reported MSE on ETTh1: **0.379**

Let's see how Time-CAG v1 compares!

In [ ]:
# RAFT baseline
raft_mse = 0.379

# Calculate performance difference
improvement = ((raft_mse - mse) / raft_mse) * 100
time_cag_mse = mse

print("=" * 80)
print("COMPARISON WITH RAFT")
print("=" * 80)
print(f"RAFT MSE:      {raft_mse:.6f}")
print(f"Time-CAG MSE:  {time_cag_mse:.6f}")
print()

if time_cag_mse < raft_mse:
    print(f"✅ SUCCESS! Time-CAG beats RAFT by {improvement:.2f}%")
    print(f"   Improvement: {raft_mse - time_cag_mse:.6f}")
else:
    print(f"❌ Time-CAG is {abs(improvement):.2f}% worse than RAFT")
    print(f"   Difference: +{time_cag_mse - raft_mse:.6f}")

print("=" * 80)

# Create comparison chart
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
models = ['RAFT\n(Retrieval)', 'Time-CAG v1\n(Long Context)']
mse_values = [raft_mse, time_cag_mse]
colors = ['#2ecc71' if time_cag_mse < raft_mse else '#e74c3c', '#3498db']

bars = ax.bar(models, mse_values, color=[colors[0], colors[1]], alpha=0.8, edgecolor='black', linewidth=2)

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, mse_values)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.6f}',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('Test MSE (Lower is Better)', fontsize=13, fontweight='bold')
ax.set_title('Experiment 1: Time-CAG v1 vs RAFT\nETTh1 Dataset (96-step forecast)', 
             fontsize=15, fontweight='bold', pad=20)
ax.set_ylim(0, max(mse_values) * 1.2)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('exp1_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ Comparison chart saved to: exp1_comparison.png")

## 9️⃣ Summary and Next Steps

**Experiment 1 Complete! 🎉**

This baseline establishes Time-CAG's performance with seq_len=720 timesteps.

**Next Steps:**
1. Run Experiment 2 (medium context: 1440 timesteps)  
2. Run Experiment 3 (longer context: 3000 timesteps)
3. Run ablation studies (experiments 4-6)
4. Run optimized configuration (experiment 7)
5. Generate final comparison graphs
6. Write workshop paper

**Download the results archive below to continue analysis locally.**

In [ ]:
# Create archive of results for download
import os

results_archive = 'exp1_outputs.zip'
os.system(f'zip -r {results_archive} experiments/results/ exp1_comparison.png checkpoints/')

print("=" * 80)
print("EXPERIMENT COMPLETE!")
print("=" * 80)
print(f"📦 Results archived to: {results_archive}")
print(f"📊 Comparison chart: exp1_comparison.png")
print(f"💾 Model checkpoint: ./checkpoints/")
print(f"📋 JSON results: {results_file}")
print("=" * 80)
print("\n💡 Download exp1_outputs.zip to your local machine for analysis")
